# F10 - Does quality filtering help separate regular from fast spiking?

Nothing is recomputed here. The firing metrics already exist for every unit from
`SUA_NATIM_firing_prop_pkl_create.py`; this notebook joins them to the inclusion
list from `F9` and asks whether cleaning sharpens the RS/FS distinction.

**The question.** Before cleaning, the evidence was split. Narrow Shallow fired
more regularly and burst less than Wide, consistent with enrichment for
fast-spiking cells. But it also fired *slower*, which is backwards: high rate is
the most reliably reported property of fast-spiking interneurons. Multi-unit
contamination was a candidate explanation for that contradiction, because
merging inflates rate, raises burst index, and pushes LV toward 1.

**What would count as help.** Four things, tested in order:

1. The rate contradiction resolves, or at least weakens
2. Effect sizes on the regularity and burst measures grow
3. The LV bimodality within Narrow Shallow collapses, i.e. it was contamination
4. A coherent fast-spiking phenotype appears: units that are simultaneously
   regular, non-bursty and fast

**What would count as no help.** Effect sizes unchanged or shrinking, the rate
contradiction persisting, and no coherent phenotype emerging. That outcome is
equally worth knowing, and it would mean the waveform classes simply do not
separate on firing regime in this dataset.

In [ ]:
import os
os.chdir('/CSNG/studekat/ripple_paper_clean_copy/code_new_filter')

In [ ]:
from functions_analysis import *
import pandas as pd, numpy as np, yaml, pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from statsmodels.stats.multitest import multipletests
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
MAIN_FOLDER = '/CSNG/studekat/ripple_paper_clean_copy'
with open(f"{MAIN_FOLDER}/code_new_filter/params_analysis.yml") as f:
    params_analysis = yaml.safe_load(f)

DF_FOLDER = f'{MAIN_FOLDER}/dataframes_new_filter'
MONKEY_LIST = ['N','F']
TYPE_REC = 'NATIM'
AREA = 'V12'
FINAL_CLASSES = params_analysis['final_classes']
CLASS_COLORS = params_analysis['colors_class']
CLASS_NAMES = params_analysis['classes_names']
NAME_MAP = dict(zip(FINAL_CLASSES, CLASS_NAMES))
CLASS_DICT = {'DOWN_narrow_shallow':'NarrBI','DOWN_narrow_sharp':'NarrTRI',
              'DOWN_wide':'Wide','DOWN_medium_shallow':'MedBI',
              'DOWN_medium_sharp':'MedTRI','UP':'Pos'}
KEY = ['monkey','date','array','cell_name']
classes_no_up = [c for c in FINAL_CLASSES if c != 'UP']

# the planned contrast
FS_CAND = 'DOWN_narrow_shallow'
RS_CAND = 'DOWN_wide'

## 1. Load: firing metrics, waveform classes, inclusion list

In [ ]:
def load_firing_df(monkey_list, params, df_folder):
    dfs = []
    for monkey in monkey_list:
        for date in params['dates'][monkey]['NATIM']:
            p = f'{df_folder}/sua_firing_NATIM/monkey{monkey}_all_arrays_date_{date}.pkl'
            try:
                with open(p,'rb') as f: dfs.append(pickle.load(f))
            except Exception: print(f'missing firing: {monkey} {date}')
    if not dfs: raise RuntimeError('no firing dataframes found')
    return pd.concat(dfs, ignore_index=True)

def load_prop_df_tagged(monkey_list, params, df_folder):
    dfs = []
    for monkey in monkey_list:
        for date in params['dates'][monkey]['NATIM']:
            p = f'{df_folder}/sua_prop_all_NATIM/monkey{monkey}_all_arrays_date_{date}.pkl'
            try:
                with open(p,'rb') as f: d = pickle.load(f)
                d['monkey'] = monkey; d['date'] = date
                dfs.append(d)
            except Exception: pass
    df = pd.concat(dfs, ignore_index=True)
    df['area_merged'] = [a if a in ['V4','IT'] else 'V12' for a in df['area']]
    df = df[~df['ch_is_noisy_100Hz']]
    df = df[~df['ch_is_noisy_120Hz']]
    return df.reset_index(drop=True)
"""
df_fire = load_firing_df(MONKEY_LIST, params_analysis, DF_FOLDER)
df_wf   = load_prop_df_tagged(MONKEY_LIST, params_analysis, DF_FOLDER)
with open(f'{DF_FOLDER}/sua_quality_{TYPE_REC}/unit_inclusion_list.pkl','rb') as f:
    df_qual = pickle.load(f)
print('firing:', df_fire.shape, '| waveform:', df_wf.shape, '| quality:', df_qual.shape)
"""

In [ ]:
df_fire = load_firing_df(MONKEY_LIST, params_analysis, DF_FOLDER)
df_wf   = load_prop_df_tagged(MONKEY_LIST, params_analysis, DF_FOLDER)
with open(f'{DF_FOLDER}/sua_quality_{TYPE_REC}/unit_inclusion_list.pkl','rb') as f:
    df_qual = pickle.load(f)
print('firing:', df_fire.shape, '| waveform:', df_wf.shape, '| quality:', df_qual.shape)

In [ ]:
wfcols = KEY + [c for c in ['final_class','area_merged','width_wf'] if c in df_wf.columns]
df = df_fire.merge(df_wf[wfcols], on=KEY, how='inner')
df = df[df['area_merged'] == AREA]

qcols = KEY + ['n_quality_pass'] + [c for c in df_qual.columns if c.startswith('pass_')]
qcols = [c for c in qcols if c in df_qual.columns]
df = df.merge(df_qual[qcols], on=KEY, how='left')

LEVELS = sorted([c for c in df.columns if c.startswith('pass_k')])
for c in LEVELS:
    df[c] = df[c].fillna(False).astype(bool)
if 'include_unit' in df.columns:
    df = df[df['include_unit']]

print(f'{df.shape[0]} units with firing metrics, class and quality')
print()
print('retention by level:')
print(f'  {"all":10s} {df.shape[0]:6d}')
for c in LEVELS:
    print(f'  {c:10s} {int(df[c].sum()):6d} ({100*df[c].mean():5.1f}%)')

In [ ]:
METRICS = ['FR_baseline','FR_transient','FR_peak_evoked','modulation_index',
           'LV_evoked','LV_baseline','CV2_evoked','CV_ISI_evoked',
           'burst_index_evoked','frac_spikes_in_burst_evoked',
           'spikes_per_burst_evoked','first_spike_latency_s',
           'first_spike_jitter_s','psth_decay_ratio']
METRICS = [m for m in METRICS if m in df.columns]
LABELS = {'FR_baseline':'Baseline FR','FR_transient':'Transient FR',
          'FR_peak_evoked':'Peak evoked FR','LV_evoked':'LV (evoked)',
          'CV2_evoked':'CV2 (evoked)','burst_index_evoked':'Burst index',
          'frac_spikes_in_burst_evoked':'Frac spikes in burst',
          'spikes_per_burst_evoked':'Spikes per burst'}

def rank_biserial(x, y):
    x, y = np.asarray(x), np.asarray(y)
    if len(x)==0 or len(y)==0: return np.nan
    u = stats.mannwhitneyu(x, y, alternative='two-sided').statistic
    return 2*u/(len(x)*len(y)) - 1

# the comparison sets: unfiltered plus every k level
SETS = [('all', df)] + [(c, df[df[c]]) for c in LEVELS]
print('comparison sets:', [(n, d.shape[0]) for n, d in SETS])

In [ ]:
# where do the 359 units go?
df_wf_area = df_wf[df_wf['area_merged'] == AREA]
print(f'waveform, V12:            {df_wf_area.shape[0]}')

merged = df_fire.merge(df_wf_area[KEY + ['final_class','area_merged']],
                       on=KEY, how='inner')
print(f'after inner-join firing:  {merged.shape[0]}   '
      f'(lost {df_wf_area.shape[0] - merged.shape[0]})')

if 'include_unit' in merged.columns:
    print(f'after include_unit:       {int(merged["include_unit"].sum())}   '
          f'(lost {int((~merged["include_unit"]).sum())})')

# which recordings are missing entirely from the firing set
have = set(map(tuple, df_fire[['monkey','date']].drop_duplicates().values))
want = set(map(tuple, df_wf_area[['monkey','date']].drop_duplicates().values))
print('\nrecordings with no firing pickle:', sorted(want - have))
for mk, dt in sorted(want - have):
    n = ((df_wf_area['monkey']==mk) & (df_wf_area['date']==dt)).sum()
    print(f'   {mk} {dt}: {n} units in V12')

## 2. Test 1 - does the rate contradiction resolve?

Fast-spiking interneurons fire faster than pyramidal cells. Before cleaning,
Narrow Shallow fired slower than Wide. A positive effect means Narrow Shallow is
faster, which is what the hypothesis predicts.

In [ ]:
rows = []
for name, d in SETS:
    a = d[d['final_class']==FS_CAND]
    b = d[d['final_class']==RS_CAND]
    r = {'set': name, 'n_FS': len(a), 'n_RS': len(b)}
    for m in ['FR_baseline','FR_transient','FR_peak_evoked']:
        if m not in d.columns: continue
        x, y = a[m].dropna(), b[m].dropna()
        r[f'{m}_FS'] = round(np.median(x),2) if len(x) else np.nan
        r[f'{m}_RS'] = round(np.median(y),2) if len(y) else np.nan
        r[f'{m}_rb'] = round(rank_biserial(x,y),3) if len(x)>=10 and len(y)>=10 else np.nan
    rows.append(r)
df_rate = pd.DataFrame(rows)
print('Narrow Shallow vs Wide. Positive rb = Narrow Shallow FASTER (hypothesis).')
print()
print(df_rate.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7,4), dpi=120)
x = np.arange(len(SETS))
for m, c in [('FR_baseline','tab:blue'),('FR_transient','tab:orange'),
             ('FR_peak_evoked','tab:green')]:
    col = f'{m}_rb'
    if col in df_rate.columns:
        ax.plot(x, df_rate[col], 'o-', color=c, label=m)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xticks(x); ax.set_xticklabels(df_rate['set'], rotation=30, fontsize=8)
ax.set_ylabel('rank-biserial (NarrBI vs Wide)')
ax.set_title('Above zero = fast-spiking direction')
ax.legend(fontsize=8, frameon=False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Test 2 - do the regularity and burst effects strengthen?

These were the measures supporting the hypothesis, and the ones most vulnerable
to contamination. A *negative* effect means Narrow Shallow is more regular and
less bursty, which is the fast-spiking direction for these measures.

In [ ]:
rows = []
for name, d in SETS:
    a = d[d['final_class']==FS_CAND]
    b = d[d['final_class']==RS_CAND]
    for m in METRICS:
        x, y = a[m].dropna(), b[m].dropna()
        if len(x) >= 10 and len(y) >= 10:
            rows.append({'set': name, 'metric': m,
                         'median_FS': np.median(x), 'median_RS': np.median(y),
                         'rb': rank_biserial(x, y),
                         'p': stats.mannwhitneyu(x, y).pvalue,
                         'n_FS': len(x), 'n_RS': len(y)})
df_eff = pd.DataFrame(rows)
piv = df_eff.pivot(index='metric', columns='set', values='rb')
piv = piv[[n for n,_ in SETS if n in piv.columns]]
print('rank-biserial, Narrow Shallow vs Wide, per cleaning level:')
print(piv.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8,6), dpi=120)
piv.plot.barh(ax=ax, alpha=0.9, colormap='viridis')
ax.axvline(0, color='k', lw=1)
ax.set_xlabel('rank-biserial (NarrBI vs Wide)'); ax.set_ylabel('')
ax.legend(fontsize=8, frameon=False, title='level')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# did effects grow?
base = piv['all']
print('change in |effect| from unfiltered:')
for c in piv.columns[1:]:
    delta = (piv[c].abs() - base.abs())
    print(f'  {c:10s} mean {delta.mean():+.3f}, '
          f'grew for {int((delta>0).sum())}/{len(delta)} metrics')

## 4. Test 3 - does the LV bimodality collapse?

If the second mode in Narrow Shallow was contamination, cleaning should leave a
single component.

In [ ]:
GVAR = 'LV_evoked'
rows = []
for name, d in SETS:
    for cl in [FS_CAND, RS_CAND]:
        x = d.loc[d['final_class']==cl, GVAR].dropna().values.reshape(-1,1)
        if len(x) < 50: continue
        bics = [GaussianMixture(k, random_state=0, n_init=3).fit(x).bic(x) for k in [1,2,3]]
        kbest = [1,2,3][int(np.argmin(bics))]
        rows.append({'set': name, 'class': CLASS_DICT[cl], 'n': len(x),
                     'k_best': kbest, 'BIC_gain_1v2': round(bics[0]-bics[1],1)})
df_bim = pd.DataFrame(rows)
print('BIC_gain_1v2 > 0 means two components preferred over one.')
print()
print(df_bim.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, len(SETS), figsize=(3.2*len(SETS), 3.2),
                         dpi=110, sharey=True)
if len(SETS)==1: axes=[axes]
for ax,(name,d) in zip(axes, SETS):
    for cl,c in [(FS_CAND,'maroon'),(RS_CAND,'tab:blue')]:
        v = d.loc[d['final_class']==cl, GVAR].dropna()
        if len(v) < 30: continue
        sns.kdeplot(v, ax=ax, color=c, label=CLASS_DICT[cl], lw=2)
    ax.set_title(f'{name}\n(n={d.shape[0]})', fontsize=9)
    ax.set_xlabel(GVAR)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 5. Test 4 - is there a coherent fast-spiking phenotype?

The strongest possible outcome: units that are simultaneously regular,
non-bursty and fast. Defined by tercile membership so the definition does not
depend on absolute thresholds, and applied within each set separately.

In [ ]:
def fs_phenotype(d):
    """regular AND non-bursty AND fast, by within-set terciles"""
    need = ['LV_evoked','burst_index_evoked','FR_baseline']
    if not all(c in d.columns for c in need): return None
    dd = d[need + ['final_class']].dropna()
    if len(dd) < 60: return None
    lo_lv    = dd['LV_evoked']          <= dd['LV_evoked'].quantile(1/3)
    lo_burst = dd['burst_index_evoked'] <= dd['burst_index_evoked'].quantile(1/3)
    hi_fr    = dd['FR_baseline']        >= dd['FR_baseline'].quantile(2/3)
    dd['is_FS_phenotype'] = lo_lv & lo_burst & hi_fr
    return dd

rows = []
for name, d in SETS:
    dd = fs_phenotype(d)
    if dd is None: continue
    r = {'set': name, 'n': len(dd),
         'n_phenotype': int(dd['is_FS_phenotype'].sum()),
         'pct': round(100*dd['is_FS_phenotype'].mean(),2)}
    for cl in classes_no_up:
        sub = dd[dd['final_class']==cl]
        r[CLASS_DICT[cl]] = round(100*sub['is_FS_phenotype'].mean(),1) if len(sub) else np.nan
    # enrichment of the phenotype in the FS candidate class vs Wide
    a = dd.loc[dd['final_class']==FS_CAND,'is_FS_phenotype']
    b = dd.loc[dd['final_class']==RS_CAND,'is_FS_phenotype']
    if len(a) >= 10 and len(b) >= 10 and (a.sum()+b.sum()) > 0:
        ct = np.array([[a.sum(), len(a)-a.sum()],[b.sum(), len(b)-b.sum()]])
        try:
            odds, p = stats.fisher_exact(ct)
            r['OR_NarrBI_vs_Wide'] = round(odds,2); r['p'] = f'{p:.1e}'
        except Exception: pass
    rows.append(r)
df_pheno = pd.DataFrame(rows)
print('Percentage of each class showing the fast-spiking phenotype')
print('(regular AND non-bursty AND fast, within-set terciles).')
print('OR > 1 means the phenotype is enriched in Narrow Shallow relative to Wide.')
print()
print(df_pheno.to_string(index=False))

## 6. Multivariate: does class become more predictable from firing?

A single number for how much the two descriptions agree, at each cleaning level.
Chance is the balanced rate.

In [ ]:
MV = ['FR_baseline','FR_transient','FR_peak_evoked','modulation_index',
      'CV2_evoked','LV_evoked','burst_index_evoked',
      'frac_spikes_in_burst_evoked','first_spike_latency_s','first_spike_jitter_s']
MV = [m for m in MV if m in df.columns]

rows = []
for name, d in SETS:
    dd = d[d['final_class'].isin(classes_no_up)][MV + ['final_class']].dropna()
    if dd.shape[0] < 200: 
        rows.append({'set': name, 'n': dd.shape[0], 'bal_acc': np.nan}); continue
    X = dd[MV].values.copy()
    for i, m in enumerate(MV):
        if m.startswith('FR_'): X[:, i] = np.log10(X[:, i] + 0.1)
    X = StandardScaler().fit_transform(X)
    y = dd['final_class'].values
    clf = RandomForestClassifier(300, random_state=0, class_weight='balanced', n_jobs=-1)
    sc = cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                         scoring='balanced_accuracy')
    rows.append({'set': name, 'n': dd.shape[0],
                 'bal_acc': round(sc.mean(),4), 'sd': round(sc.std(),4),
                 'chance': round(1/len(np.unique(y)),4)})
df_clf = pd.DataFrame(rows)
print(df_clf.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5), dpi=120)
ok = df_clf['bal_acc'].notna()
ax.plot(range(ok.sum()), df_clf.loc[ok,'bal_acc'], 'o-', color='maroon')
if 'chance' in df_clf.columns and ok.any():
    ax.axhline(df_clf.loc[ok,'chance'].iloc[0], color='k', ls='--', lw=1, label='chance')
ax.set_xticks(range(ok.sum())); ax.set_xticklabels(df_clf.loc[ok,'set'], rotation=30, fontsize=8)
ax.set_ylabel('balanced accuracy'); ax.legend(fontsize=8, frameon=False)
ax.set_title('Predicting waveform class from firing metrics')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Verdict

In [ ]:
print('='*66)
print('DID CLEANING HELP SEPARATE REGULAR FROM FAST SPIKING?')
print('='*66)

best = LEVELS[-1] if LEVELS else 'all'
def get(dfx, col, s):
    r = dfx[dfx['set']==s]
    return r[col].iloc[0] if len(r) and col in r.columns else np.nan

print(f'\ncomparing "all" against "{best}"\n')

print('1. RATE CONTRADICTION (positive rb = fast-spiking direction)')
for m in ['FR_baseline','FR_transient','FR_peak_evoked']:
    c = f'{m}_rb'
    a, b = get(df_rate,c,'all'), get(df_rate,c,best)
    if np.isfinite(a) and np.isfinite(b):
        verdict = 'RESOLVED' if b > 0 > a else ('weakened' if abs(b) < abs(a) else 'persists')
        print(f'   {m:18s} {a:+.3f} -> {b:+.3f}   {verdict}')

print('\n2. REGULARITY / BURST EFFECTS')
if best in piv.columns:
    d = (piv[best].abs() - piv['all'].abs())
    print(f'   |effect| grew for {int((d>0).sum())}/{len(d)} metrics, mean {d.mean():+.3f}')

print('\n3. LV BIMODALITY')
for cl in [FS_CAND, RS_CAND]:
    n = CLASS_DICT[cl]
    r0 = df_bim[(df_bim['set']=='all') & (df_bim['class']==n)]
    r1 = df_bim[(df_bim['set']==best) & (df_bim['class']==n)]
    if len(r0) and len(r1):
        print(f'   {n:8s} k={r0["k_best"].iloc[0]} -> k={r1["k_best"].iloc[0]}')

print('\n4. FAST-SPIKING PHENOTYPE (odds ratio NarrBI vs Wide)')
a, b = get(df_pheno,'OR_NarrBI_vs_Wide','all'), get(df_pheno,'OR_NarrBI_vs_Wide',best)
if np.isfinite(a) and np.isfinite(b):
    print(f'   OR {a:.2f} -> {b:.2f}')

print('\n5. CLASS PREDICTABILITY')
a, b = get(df_clf,'bal_acc','all'), get(df_clf,'bal_acc',best)
if np.isfinite(a) and np.isfinite(b):
    print(f'   balanced accuracy {a:.3f} -> {b:.3f}')
print('='*66)

In [ ]:
out = f'{DF_FOLDER}/sua_quality_{TYPE_REC}/rs_fs_check'
ensure_dir_exists(out)
df_rate.to_csv(f'{out}/rate_contrast.csv', index=False)
piv.to_csv(f'{out}/effect_sizes_by_level.csv')
df_bim.to_csv(f'{out}/lv_bimodality.csv', index=False)
df_pheno.to_csv(f'{out}/fs_phenotype.csv', index=False)
df_clf.to_csv(f'{out}/classifier.csv', index=False)
print('saved to', out)

## Dimansionality reduction

In [ ]:
# ---- Embedding of firing properties, putative L4 (Positive and Triphasic excluded) ----
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

INCLUDE_WIDE = True          # False -> NarrBI + MedBI only (strict L4)
EMB_LEVEL    = 'pass_k3'     # 'all' for unfiltered

SEL_CLASSES = ['DOWN_narrow_shallow', 'DOWN_medium_shallow']
if INCLUDE_WIDE:
    SEL_CLASSES = SEL_CLASSES + ['DOWN_wide']

EMB_METRICS = ['FR_baseline', 'FR_transient', 'FR_peak_evoked', 'modulation_index',
               'LV_evoked', 'LV_baseline', 'CV2_evoked',
               'burst_index_evoked', 'frac_spikes_in_burst_evoked',
               'spikes_per_burst_evoked', 'psth_decay_ratio',
               'first_spike_latency_s', 'first_spike_jitter_s']
EMB_METRICS = [m for m in EMB_METRICS if m in df.columns]

d_emb = df if EMB_LEVEL == 'all' else df[df[EMB_LEVEL]]
d_emb = d_emb[d_emb['final_class'].isin(SEL_CLASSES)]
d_emb = d_emb[EMB_METRICS + ['final_class', 'monkey', 'array']].dropna()
print(f'{d_emb.shape[0]} units, {len(EMB_METRICS)} metrics, level={EMB_LEVEL}')
print(d_emb['final_class'].value_counts().rename(index=CLASS_DICT).to_string())

X = d_emb[EMB_METRICS].values.astype(float)
# rate variables span orders of magnitude and are right-skewed
for i, m in enumerate(EMB_METRICS):
    if m.startswith('FR_') or m == 'spikes_per_burst_evoked':
        X[:, i] = np.log10(np.clip(X[:, i], 0, None) + 0.1)
X = StandardScaler().fit_transform(X)

pca = PCA(n_components=min(6, X.shape[1])).fit(X)
Xp = pca.transform(X)
Xt = TSNE(n_components=2, random_state=0, init='pca',
          perplexity=min(30, max(5, X.shape[0] // 20))).fit_transform(X)
print('explained variance:', np.round(pca.explained_variance_ratio_, 3))

fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=110)

# --- PCA, coloured by class ---
ax = axes[0, 0]
for cl in SEL_CLASSES:
    m = (d_emb['final_class'] == cl).values
    ax.scatter(Xp[m, 0], Xp[m, 1], s=10, alpha=0.35,
               color=CLASS_COLORS[cl], label=CLASS_DICT[cl])
ax.set_xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.0f}%)')
ax.set_ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.0f}%)')
ax.legend(fontsize=8, markerscale=2, frameon=False)
ax.set_title('PCA of firing properties')

# --- loadings, to read the axes ---
ax = axes[0, 1]
load = pd.DataFrame(pca.components_[:3].T, index=EMB_METRICS,
                    columns=['PC1', 'PC2', 'PC3'])
sns.heatmap(load, cmap='RdBu_r', center=0, ax=ax,
            cbar_kws={'label': 'loading'}, annot=False)
ax.set_title('PC loadings')
ax.tick_params(labelsize=7)

# --- tSNE ---
ax = axes[0, 2]
for cl in SEL_CLASSES:
    m = (d_emb['final_class'] == cl).values
    ax.scatter(Xt[m, 0], Xt[m, 1], s=10, alpha=0.35,
               color=CLASS_COLORS[cl], label=CLASS_DICT[cl])
ax.set_xlabel('tSNE 1'); ax.set_ylabel('tSNE 2')
ax.set_title('tSNE of firing properties')

# --- PCA coloured by three key metrics, to see what the structure IS ---
for ax, m in zip(axes[1], ['FR_baseline', 'LV_evoked', 'burst_index_evoked']):
    if m not in d_emb.columns:
        ax.axis('off'); continue
    v = d_emb[m].values
    if m.startswith('FR_'):
        v = np.log10(np.clip(v, 0, None) + 0.1)
    sc = ax.scatter(Xp[:, 0], Xp[:, 1], c=v, s=10, alpha=0.5, cmap='viridis')
    plt.colorbar(sc, ax=ax, label=('log10 ' if m.startswith('FR_') else '') + m)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(f'coloured by {m}')

for ax in axes.flat:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# --- is the class separation better than chance? ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
y = d_emb['final_class'].values
clf = RandomForestClassifier(400, random_state=0, class_weight='balanced', n_jobs=-1)
sc = cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                     scoring='balanced_accuracy')
print(f'\nbalanced accuracy {sc.mean():.3f} +/- {sc.std():.3f}   '
      f'chance {1/len(np.unique(y)):.3f}')

# --- how separated are the class centroids, relative to within-class scatter? ---
print('\ncentroid separation in the first 3 PCs (units of pooled within-class SD):')
cent = {cl: Xp[(d_emb['final_class']==cl).values, :3].mean(axis=0) for cl in SEL_CLASSES}
pooled = np.mean([Xp[(d_emb['final_class']==cl).values, :3].std(axis=0)
                  for cl in SEL_CLASSES], axis=0)
for i, a in enumerate(SEL_CLASSES):
    for b in SEL_CLASSES[i+1:]:
        dist = np.linalg.norm((cent[a]-cent[b]) / pooled)
        print(f'  {CLASS_DICT[a]:7s} vs {CLASS_DICT[b]:7s}: {dist:.2f}')

## How to read the verdict

**Cleaning helped** if the rate contradiction resolves or weakens, the
regularity and burst effect sizes grow, the LV bimodality collapses to one
component, and the phenotype odds ratio rises. Those four moving together would
mean contamination was masking a real RS/FS distinction.

**Cleaning did not help** if effect sizes are flat or shrinking and the rate
contradiction persists. Note that shrinking effects are expected to some degree
purely from reduced sample size, so read the medians alongside the effect sizes
rather than the significance.

**The most likely outcome, on the evidence so far**, is partial. The firing rate
figures already showed Medium Biphasic collapsing from about 52 to 17 Hz while
Narrow Shallow stayed below Wide. If that pattern holds here, contamination
explained the Medium Biphasic anomaly but not the Narrow Shallow rate
contradiction, and the defensible conclusion stays as it was: Narrow Shallow is
enriched for fast-spiking cells on regularity and bursting, but the rate
evidence points the other way.

**Two caveats that do not go away with cleaning.** Units recorded across
multiple blocks within a day still enter every test repeatedly, so effect sizes
rather than p-values are the quantity to interpret. And none of these metrics
speak to synaptic sign: fast-spiking is enriched for parvalbumin-positive cells,
not equivalent to inhibitory identity.